# Demo — defineutils and Other Command-Line Tools

The checks you built by hand in Blocks 1–3, productized: defineutils is a pip-installable set
of Define-XML command-line utilities with odmlib as its engine. This notebook runs each
command from shell cells (the `!` prefix runs a shell command; `{sys.executable}` pins the
commands to this notebook's Python environment).

See `lecture_notes.md` for the command reference and the wider tool ecosystem.

In [1]:
import os
import sys

os.makedirs("output", exist_ok=True)
py = sys.executable
print(py)

/home/sam/src/r-pharma-odmlib/.venv/bin/python


## 1. metrics — describe a define.xml

Block 1's exercise as a command. First on the minimal DM define from Block 2:

In [2]:
!{py} -m defineutils.metrics -d ../data/define_dm_example.xml

Define-XML metrics
  File:     ../data/define_dm_example.xml
  Size:     2,767 bytes (2.7 KiB)
  Created:  2026-08-19T12:00:00  (ODM/@CreationDateTime)
  Modified: 2026-08-19T16:56:03  (file system)
  Model:    Define-XML v2.1 (odmlib define_2_1, odmlib 0.2.1)

STUDY
  Study name:        RPH2026
  Protocol name:     RPH-2026-001
  Description:       R/Pharma 2026 odmlib workshop study
  Study OID:         ST.RPH2026
  MetaDataVersion:   MDV.RPH2026.1
                     RPH2026 Data Definitions
  Define version:    2.1.0
  File OID:          DEF.RPH2026.DM
  Context:           Submission
  Originator:        R/Pharma 2026 Workshop
  Source system:     odmlib
  Standards (1):     SDTMIG 3.4 IG (Final)

ELEMENT COUNTS
  Document
    ODM                  1
    Study                1
    GlobalVariables      1
    MetaDataVersion      1
    StudyName            1
    StudyDescription     1
    ProtocolName         1
  MetaDataVersion definitions
    def:Standards        1
    def:Annotate

The same command scales to the full MSG example — here with the per-dataset table suppressed
to keep the output short (drop `--no-datasets` to see every dataset's variables):

In [3]:
!{py} -m defineutils.metrics -d ../data/defineV21-SDTM.xml --no-datasets

Define-XML metrics
  File:     ../data/defineV21-SDTM.xml
  Size:     173,897 bytes (169.8 KiB)
  Created:  2019-02-11T15:30:01  (ODM/@CreationDateTime)
  Modified: 2026-08-19T16:40:42  (file system)
  Model:    Define-XML v2.1 (odmlib define_2_1, odmlib 0.2.1)

STUDY
  Study name:        CDISC01_1
  Protocol name:     CDISC01-1
  Description:       CDISC Test Study Modified to illustrate Define-XML 2.1 features
  Study OID:         STDY.www.cdisc.org.CDISC01_1
  MetaDataVersion:   MDV.CDISC01_1.1.SDTMIG.3.1.2.SDTM.1.2_X
                     Study CDISC01_1, Data Definitions V-1
  Define version:    2.1.0
  File OID:          www.cdisc.org/StudyCDISC01_1/1/Define-XML_2.1.0
  Context:           Other
  Originator:        CDISC Data Exchange Standards Team
  Source system:     M.Hungria-System 2.1-A1
  Standards (5):     SDTMIG 3.1.2 IG (Final), SDTMIG 3.2 IG (Final), SDTMIG-MD 1.0 IG (Final),
                     CDISC/NCI SDTM 2011-12-09 CT (Final), CDISC/NCI SDTM 2015-12-18 CT (Final)

## 2. validate — XSD schema validation

Block 3, layer 1 as a command. On the broken file from the exercise (`-L 0` lists all
locations). Exit codes make this CI-ready: 0 clean, 1 findings, 2 could-not-run.

In [4]:
!{py} -m defineutils.validate -d ../data/defineV21-SDTM-invalid.xml -L 0

Define-XML schema validation
  File:   ../data/defineV21-SDTM-invalid.xml
  Schema: /home/sam/src/r-pharma-odmlib/.venv/lib/python3.12/site-packages/defineutils/validate/schema/cdisc-define-2.1/define2-1-0.xsd
  Scope:  1 error in 1 distinct problem

ERRORS (1)

  attribute_error  ODM   (1 occurrence)
      missing required attribute '{http://www.cdisc.org/ns/def/v2.1}Context'
      - line 28  (document root)

SUMMARY
  1 error in ../data/defineV21-SDTM-invalid.xml
  Schema: /home/sam/src/r-pharma-odmlib/.venv/lib/python3.12/site-packages/defineutils/validate/schema/cdisc-define-2.1/define2-1-0.xsd


There's the missing `def:Context` you diagnosed in Block 3 — with a line number. Machine-readable
output for pipelines:

In [5]:
!{py} -m defineutils.validate -d ../data/defineV21-SDTM-invalid.xml --json

{
  "define_file": "../data/defineV21-SDTM-invalid.xml",
  "schema_file": "/home/sam/src/r-pharma-odmlib/.venv/lib/python3.12/site-packages/defineutils/validate/schema/cdisc-define-2.1/define2-1-0.xsd",
  "xmlschema_version": "4.3.2",
  "valid": false,
  "counts": {
    "errors": 1,
    "findings": 1
  },
  "findings": [
    {
      "check": "attribute_error",
      "severity": "error",
      "element": "ODM",
      "reason": "missing required attribute '{http://www.cdisc.org/ns/def/v2.1}Context'",
      "message": "ODM: missing required attribute '{http://www.cdisc.org/ns/def/v2.1}Context'",
      "detail": "missing required attribute '{http://www.cdisc.org/ns/def/v2.1}Context'",
      "locations": [
        {
          "path": "(document root)",
          "line": 28
        }
      ]
    }
  ]
}


## 3. definerefs — OID reference/definition integrity

Block 3, layer 2 as a command — run against the broken-refs file you repaired in the
exercise. It finds the dangling reference *and* the orphan in one pass:

In [6]:
!{py} -m defineutils.definerefs -d ../data/define_broken_refs.xml -L 0

Define-XML OID reference/definition check
  File:  ../data/define_broken_refs.xml
  Model: Define-XML v2.1 (odmlib define_2_1, odmlib 0.2.1)
  Scope: 11 definitions, 8 references across 4 checked attributes

ERRORS (1)

  undefined_reference  ItemOID = "IT.DM.SEXX"   (1 occurrence)
      expects an ItemDef; no element defines this OID
      - ItemRef in MetaDataVersion[MDV.RPH2026.1]/ItemGroupDef[IG.DM]

WARNINGS (2)

  orphan_definition  IT.DM.ARMCD
      ItemDef defined but never referenced (expected via ItemOID)
      - ItemDef in MetaDataVersion[MDV.RPH2026.1]

  orphan_definition  IT.DM.SEX
      ItemDef defined but never referenced (expected via ItemOID)
      - ItemDef in MetaDataVersion[MDV.RPH2026.1]

SUMMARY
  1 error, 2 warnings in ../data/define_broken_refs.xml
  Checked attributes: ArchiveLocationID, CodeListOID, ItemOID, StandardOID
  Skipped attributes: FileOID, PriorFileOID, StudyOID, MetaDataVersionOID, ItemGroupOID
                      (file-level or structurally gua

## 4. definehtml — the browsable HTML view

The standard Define-XML stylesheet, applied in one command. Open the result from the file
browser to see the familiar reviewer's view of the define you built in Block 2:

In [ ]:
!{py} -m defineutils.definehtml -d ../data/define_dm_example.xml -o output/define_dm.html
print("wrote output/define_dm.html -", os.path.getsize("output/define_dm.html"), "bytes")

## 5. definepp — pretty-print

Byte-careful re-indentation (comments, namespaces, and content preserved):

In [7]:
!{py} -m defineutils.definepp -d ../data/define_dm_example.xml -o output/define_dm_pretty.xml

with open("output/define_dm_pretty.xml") as f:
    print("".join(f.readlines()[:20]))

<?xml version='1.0' encoding='UTF-8'?>
<ODM xmlns="http://www.cdisc.org/ns/odm/v1.3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:def="http://www.cdisc.org/ns/def/v2.1" FileOID="DEF.RPH2026.DM" FileType="Snapshot" CreationDateTime="2026-08-19T12:00:00" ODMVersion="1.3.2" def:Context="Submission" Originator="R/Pharma 2026 Workshop" SourceSystem="odmlib">
  <Study OID="ST.RPH2026">
    <GlobalVariables>
      <StudyName>RPH2026</StudyName>
      <StudyDescription>R/Pharma 2026 odmlib workshop study</StudyDescription>
      <ProtocolName>RPH-2026-001</ProtocolName>
    </GlobalVariables>
    <MetaDataVersion OID="MDV.RPH2026.1" Name="RPH2026 Data Definitions" Description="Demographics metadata for the workshop" def:DefineVersion="2.1.0">
      <def:Standards>
        <def:Standard OID="STD.1" Name="SDTMIG" Type="IG" Version="3.4" Status="Final"/>
      </def:Standards>
      <ItemGroupDef OID="IG.DM" Name="DM" Repeating="No" IsReferenceData="No" SASDatasetName="DM" Domain="DM" Purpose

## The point

`metrics` ≈ your Block 1 notebook. `validate` ≈ Block 3 layer 1. `definerefs` ≈ Block 3
layer 2. Each is an odmlib program wrapped in argparse — after today, the distance from your
notebook code to a tool like these is short.

More odmlib-based tools (links in `lecture_notes.md`): **dfine** (Define ↔ Excel),
**gendefine** (spreadsheet → Define-XML), **dsjconvert/dsjdiff** (Dataset-JSON conversion and
comparison), **define-mcp-server** (Define-XML for AI agents), and the **odmlib_examples** /
**odmlib_snippets** reference repos.